# Roman Light Curves Fitting


### 1. Imports 


In [1]:
from astropy.io import fits
from astropy.cosmology import FlatLambdaCDM, LambdaCDM, FlatwCDM, w0waCDM
from astropy import units as u
from astropy.table import Table

import pandas as pd
from pandas import DataFrame

import matplotlib.pyplot as plt
import math

from numpy import log10 as log10
from numpy import sqrt as sqrt
from numpy import square as square
from numpy import *
import numpy as np

import os
import glob

import csv
import random
import sncosmo


import scipy.stats as st
import scipy.optimize as opt
from scipy.interpolate import interp1d

from mpfit import mpfit


import warnings
warnings.filterwarnings("ignore")

### 2. Load Configuration & Inputs




In [2]:
from config import config
cfg = config.fitting
random.seed(cfg.seed)
np.random.seed(cfg.seed)


folder = os.path.expanduser(cfg.sim_dir) 

head_files = sorted(glob.glob(os.path.join(folder, cfg.head_pattern)))
phot_files = [f.replace("HEAD.FITS", "PHOT.FITS") for f in head_files]


file_pairs = list(zip(head_files, phot_files))

print(f"Found {len(file_pairs)} HEAD/PHOT file pairs")

eigen_val=np.genfromtxt(cfg.eigen_val_path)

os.makedirs(cfg.tables_dir, exist_ok=True)
os.makedirs(cfg.plots_dir, exist_ok=True)
os.makedirs(os.path.dirname(cfg.long_csv), exist_ok=True)

Found 90 HEAD/PHOT file pairs


### 3. Plot Settings

In [3]:
color=['#000000', '#E69F00', '#56B4E9', '#009E73',
            '#CC79A7', '#0072B2', '#D55E00',  '#F0E442']

dict_color={'R':color[0], 'Z': color[1], 'Y':color[2], 'J':color[3], 'H':color[4], 'F':color[5]}

dict_number={'R':-2.0, 'Z':-1.0, 'Y':0.0, 'J':1.0,  'H':2.0,  'F':3.0}
dict_sym={'R':'^', 'Z':'h',  'Y':'P', 'J':'o', 'H':'*', 'F':'s'}
dict_legend={'R':'R-2.0', 'Z':'Z-1.0', 'Y':'Y',  'J':'J+1.0', 'H':'H+2.0', 'F':'F+3.0'}
dict_ms={ 'R':6, 'Z':6, 'Y':6, 'J':5, 'H':8, 'F':4}


### 4. FPCA Model and Utility Functions

In [4]:
ph=eigen_val[1:,0]
vec0=eigen_val[1:,1]
vec1=eigen_val[1:,2]
vec2=eigen_val[1:,3]


imod0 = interp1d(ph, vec0, fill_value='extrapolate')
imod1 = interp1d(ph, vec1, fill_value='extrapolate')
imod2 = interp1d(ph, vec2, fill_value='extrapolate')


def leng_calc1(time_array2, t_o2, redshift):
    mphase = (time_array2-t_o2)/(1+redshift)
    indsel=(np.where((mphase>=-10.0) & (mphase<=40.0)))[0]
    return len(indsel)


def modelcurve1(x_calc, theta, redshift):
    t_min, mag_o, beta1, beta2 = theta
    mphase = (x_calc-t_min)/(1+redshift)

    if mphase >= -10.0 and mphase <= 40.0:
        VecP0 = float(imod0(mphase))
        VecP1 = float(imod1(mphase))
        VecP2 = float(imod2(mphase))

        y_calc = mag_o + VecP0 + beta1*VecP1 + beta2*VecP2
    
    else:
        y_calc=np.nan
          
    return y_calc


def myfunctlin1(p, fjac=None, x=None, y=None, err=None, ini_params=None, redshift=None):
    
    dev = np.zeros((len(x)), dtype = np.float64)
    
    for kk in range(len(dev)):
        
        y_calc=modelcurve1(x[kk], p, redshift)
        
        if not np.isnan(y_calc):
            dev[kk]=(y[kk]-y_calc)/err[kk]
  
    status = 0
    return [status, dev]


def mpfit_run_flux1(time_array, data_vec, data_vec_unc, ini_params, tym_cons_low, tym_cons_high, redshift):

    leng=leng_calc1(time_array, ini_params[0], redshift)
    if(leng<cfg.min_point):
        return np.zeros((4)), np.zeros((4)), 10000.0, 0
    
    p0=ini_params

    parinfo=[]

    for i in range(len(p0)):
        parinfo.append({'value':0., 'fixed':0, 'limited':[0,0], 'limits':[0.,0.]})
    for i in range(len(p0)): 
        parinfo[i]['value']=p0[i]

    parinfo[0]['limited'][0] = 1
    parinfo[0]['limits'][0]  = tym_cons_low
    parinfo[0]['limited'][1] = 1
    parinfo[0]['limits'][1]  = tym_cons_high

    parinfo[2]['limited'][0] = 1
    parinfo[2]['limits'][0]  = -10
    parinfo[2]['limited'][1] = 1
    parinfo[2]['limits'][1]  = 10

    parinfo[3]['limited'][0] = 1
    parinfo[3]['limits'][0]  = -10
    parinfo[3]['limited'][1] = 1
    parinfo[3]['limits'][1]  = 10


    fa = {'x':time_array, 'y':data_vec, 'err':data_vec_unc, 'ini_params':ini_params, 'redshift': redshift}
    
    m = mpfit(myfunctlin1,  parinfo=parinfo, quiet=True, functkw=fa)

    if (m.status <= 0): 
        return np.zeros(4), np.zeros(4), 10000.0, 0
        
    else:
        chi=m.fnorm
        leng_final = leng_calc1(time_array, m.params[0], redshift)  # recompute with FITTED t_min

    return m.params, m.perror, chi, leng_final-4

### 5. Main loop over light curves

In [5]:
total_snids = set()
for hf, pf in file_pairs:
    ids = Table.read(hf, hdu=1)['SNID']
    total_snids.update(str(x).strip() for x in ids)

total_count = len(total_snids)



global_ii = 0

data_1=[]
data_2=[]

milestone = max(1, total_count // 100)
print(f"Total unique SNIDs across all files: {total_count}")

type_map = {10: 'SNIa', 11: 'SNIa-91bg', 12: 'SNIax', 30: 'CCSN', 40: 'SLSN',
            42: 'TDE', 45: 'ILOT', 50: 'KN', 59: 'PISN', 60: 'AGN'}



for hf, pf in file_pairs:
    head_cols = ['SNID', 'SNTYPE', 'REDSHIFT_FINAL', 'REDSHIFT_FINAL_ERR', 'PTROBS_MIN', 'PTROBS_MAX']
    head_df = Table.read(hf, hdu=1)[head_cols].to_pandas()
    phot_df = Table.read(pf, hdu=1).to_pandas()

    head_df['SNID'] = head_df['SNID'].str.decode('utf-8').str.strip()
    phot_df['BAND'] = phot_df['BAND'].str.decode('utf-8').str.strip().str[0]
    phot_df['FIELD'] = phot_df['FIELD'].str.decode('utf-8').str.strip().str.split('+').str[0]

    if(global_ii>=total_count):
        break

    for ii in range(len(head_df)):
        if(global_ii>=total_count):
            break
        if((global_ii%milestone)==0):
            
            pct = (global_ii / total_count) * 100
            print(f"Progress: {pct:.0f}%  ({global_ii}/{total_count})")


        data_1=[]
        table_1=[]

        row = head_df.iloc[ii]

        start = int(row['PTROBS_MIN'])-1
        end = int(row['PTROBS_MAX'])   # inclusive index

        type = type_map.get(int(head_df['SNTYPE'].iloc[ii]), str(head_df['SNTYPE'].iloc[ii]))
        obj_id=head_df['SNID'].iloc[ii]
        
       
        redshift=head_df['REDSHIFT_FINAL'].iloc[ii]
        redshift_e=head_df['REDSHIFT_FINAL_ERR'].iloc[ii]
        lc = phot_df.iloc[start:end].copy()

        count=0
        fil_idx=[]
        fil_fld=[]
        array_mag=[]
        array_time=[]
        array_err=[]

       
        
        mag1=(-2.5*log10(lc['FLUXCAL'])+31.4).to_numpy()
        mag_err1=abs(1.086*lc['FLUXCALERR']/lc['FLUXCAL']).to_numpy()
        filter= lc['BAND'].to_numpy()
        time=lc['MJD'].to_numpy()
        field_arr=lc['FIELD'].to_numpy()
        
        mask=~np.isnan(mag1)
        if config.globals.fields:
            mask &= np.isin(field_arr, config.globals.fields)
        mag1 = mag1[mask]
        
        mag_err1 = mag_err1[mask]
        filter = filter[mask]
        
        time = time[mask]
        field_arr = field_arr[mask]

        # group points per band, once, no repeated blocks
        
        band_data = {b: {'mag':[], 'err':[], 't':[], 'fld':[]} for b in config.globals.bands}
        for jj in range(len(mag1)):
            b = filter[jj]
            if b in band_data:
                band_data[b]['mag'].append(mag1[jj])
                band_data[b]['err'].append(mag_err1[jj])
                band_data[b]['t'].append(time[jj])
                band_data[b]['fld'].append(field_arr[jj])

        for b in list(config.globals.bands):
            
            arr = band_data[b]['mag']
       
            
            if arr:
                arr = np.array(arr)
                if len(arr) >= cfg.min_point:
                    count += 1
                    array_mag.append(arr)
                    array_time.append(np.array(band_data[b]['t']))
                    array_err.append(np.array(band_data[b]['err']))
                    fil_idx.append(b)
                    
                    f_list = band_data[b]['fld']
                    fil_fld.append(max(set(f_list), key=f_list.count))

        if(count>0):

            a_arr=np.zeros((len(array_mag)))
            for kkk in range(len(array_mag)):
                a_arr[kkk]=len(array_mag[kkk])

            b_arr=np.argsort(-a_arr)

       

            fil_idx = [fil_idx[ll] for ll in b_arr]
            fil_fld = [fil_fld[ll] for ll in b_arr]
            array_mag = [array_mag[ll] for ll in b_arr]
            array_time = [array_time[ll] for ll in b_arr]
            array_err = [array_err[ll] for ll in b_arr]

            for band_idx in range(count):

                if(band_idx==0):
                    chi1 = np.full(cfg.num_attempts, 10000.0)
                    final_params1=np.zeros((cfg.num_attempts,4))
                    final_errors1=np.zeros((cfg.num_attempts,4))
                    leng1=np.zeros(cfg.num_attempts)

                    final_params=np.zeros(4)
                    final_errors=np.zeros(4)

                    t_cons1=(array_time[0])[np.argmin(array_mag[0])]

                    for attempt in range(cfg.num_attempts):
                        ini_params=np.array([t_cons1+random.uniform(-cfg.jitter_time_first, cfg.jitter_time_first)*(1+redshift),
                     np.min(array_mag[band_idx])+random.uniform(-cfg.jitter_mag, cfg.jitter_mag),
                     random.uniform(-cfg.jitter_beta1, cfg.jitter_beta1), random.uniform(-cfg.jitter_beta2, cfg.jitter_beta2)])
                        final_params1[attempt,:], final_errors1[attempt,:], chi1[attempt], leng1[attempt] = mpfit_run_flux1(array_time[band_idx], array_mag[band_idx], array_err[band_idx], ini_params, t_cons1-10*(1+redshift), t_cons1+10 *(1+redshift), redshift)


                    valid = (chi1 < 10000.0) & (leng1 > 0)
                    ratio = np.full(cfg.num_attempts, 10000.0)
                    ratio[valid] = chi1[valid] / leng1[valid]
                    min_ind = np.argmin(ratio)

                    final_params[:]=final_params1[min_ind,:]
                    final_errors[:]=final_errors1[min_ind,:]
                    chi=chi1[min_ind]
                    leng=leng1[min_ind]

                    if(chi/leng < 10000.0 and chi/leng>0):
                    
                        t_cons2=final_params[0]

                        data_1.append([obj_id,  type, fil_fld[band_idx], fil_idx[band_idx], redshift, final_params[1], final_params[2], final_params[3], final_params[0],
                                        redshift_e, final_errors[1], final_errors[2], final_errors[3], final_errors[0],
                                        chi, leng])

                        data_2.append([obj_id,  type, fil_fld[band_idx], fil_idx[band_idx], redshift, final_params[1], final_params[2], final_params[3], final_params[0],
                                        redshift_e, final_errors[1], final_errors[2], final_errors[3], final_errors[0],
                                        chi, leng])

                        tmp_table = Table([array_time[band_idx], array_mag[band_idx], array_err[band_idx]],
                                        names=('time', 'mag', 'magerr'),
                                        meta={'redshift': redshift,
                                            'redshift_e': redshift_e, 'filter':fil_idx[band_idx]})
                        
                        #tmp_table.write(os.path.join(cfg.tables_dir, f"{obj_id}_{fil_idx[band_idx]}.csv"), overwrite=True)
                                              
                                                
                        table_1.append(tmp_table)
                        

                    else:
                        break

                else:

                    chi1 = np.full(cfg.num_attempts, 10000.0)
                    final_params1=np.zeros((cfg.num_attempts,4))
                    final_errors1=np.zeros((cfg.num_attempts,4))
                    leng1=np.zeros(cfg.num_attempts)

                    final_params=np.zeros(4)
                    final_errors=np.zeros(4)

                    for num_attempt in range(cfg.num_attempts):
                        ini_params=np.array([t_cons2+random.uniform(-cfg.jitter_time_other, cfg.jitter_time_other)*(1+redshift), np.min(array_mag[band_idx])+random.uniform(-cfg.jitter_mag, cfg.jitter_mag), random.uniform(-cfg.jitter_beta1, cfg.jitter_beta1), random.uniform(-cfg.jitter_beta2, cfg.jitter_beta2)])
                                           
                        final_params1[num_attempt,:], final_errors1[num_attempt,:], chi1[num_attempt], leng1[num_attempt] = mpfit_run_flux1(array_time[band_idx], array_mag[band_idx], array_err[band_idx], ini_params, t_cons2-7*(1+redshift), t_cons2+7*(1+redshift), redshift)

                    valid = (chi1 < 10000.0) & (leng1 > 0)
                    ratio = np.full(cfg.num_attempts, 10000.0)
                    ratio[valid] = chi1[valid] / leng1[valid]
                    min_ind = np.argmin(ratio)

                    final_params[:]=final_params1[min_ind,:]
                    final_errors[:]=final_errors1[min_ind,:]
                    chi=chi1[min_ind]
                    leng=leng1[min_ind]

                    if(chi/leng < 10000.0 and chi/leng>0):

                        data_1.append([obj_id, type, fil_fld[band_idx], fil_idx[band_idx], redshift, final_params[1], final_params[2], final_params[3], final_params[0],
                                        redshift_e, final_errors[1], final_errors[2], final_errors[3], final_errors[0],
                                        chi, leng])

                        data_2.append([obj_id, type, fil_fld[band_idx], fil_idx[band_idx], redshift, final_params[1], final_params[2], final_params[3], final_params[0],
                                        redshift_e, final_errors[1], final_errors[2], final_errors[3], final_errors[0],
                                        chi, leng])

                        tmp_table = Table([array_time[band_idx], array_mag[band_idx], array_err[band_idx]],
                                        names=('time', 'mag', 'magerr'),
                                        meta={ 'redshift': redshift,
                                            'redshift_e': redshift_e,  'filter':fil_idx[band_idx]})
                        #tmp_table.write(os.path.join(cfg.tables_dir, f"{obj_id}_{fil_idx[band_idx]}.csv"), overwrite=True)
                        table_1.append(tmp_table)
                        

        
            model=pd.DataFrame(data_1).to_numpy()

            legen=[]

            # if(len(model)>0 ):

            #     for j in range(len(model)):

            #         plo_data_x=eigen_val[:,0]*(1+model[j,4])+model[j,8]-model[0,8]
            #         plo_data_y=model[j,5] + eigen_val[:,1] + model[j,6]*eigen_val[:,2] + model[j,7]*eigen_val[:,3]+dict_number[model[j,3]]

            #         plt.plot(plo_data_x, plo_data_y, color=dict_color[model[j,3]],   linestyle='solid', label='_nolegend_')

            #         plt.errorbar(table_1[j]['time']-model[0,8],  table_1[j]['mag']+dict_number[model[j,3]], yerr=table_1[j]['magerr'], color=dict_color[model[j,3]], fmt=dict_sym[model[j,3]], markersize=dict_ms[model[j,3]], elinewidth=1.5, label=dict_legend[model[j,3]])

            #         coor=250
            #         plt.annotate(dict_legend[model[j,3]], xy = (plo_data_x[coor], plo_data_y[coor]), xytext = (1*plo_data_x[coor], 0.99*plo_data_y[coor]), color  = dict_color[model[j,3]])

            #         plo_data_y=np.array(plo_data_y)

            #         if(j==0):
            #             ymin=np.nanmin(plo_data_y[1:])
            #             ymax=np.nanmax(plo_data_y[1:])
            #         else:
            #             ymin=min(np.nanmin(plo_data_y[1:]), ymin)
            #             ymax=max(np.nanmax(plo_data_y[1:]), ymax)

            #     plt.xlim([-10*(1+model[0,4])-7, +40*(1+model[0,4])+7])
            #     plt.ylim([ymin-0.5, ymax+0.5])

            #     plt.xlabel('Time (Days)')
            #     plt.ylabel('Magnitude')
            #     plt.gca().invert_yaxis()

            #     rw=np.round(model[j,4],2)

            #     plt.title('ID= '+str(obj_id)+'  '   + '  '      +'z= '+ str(rw) +'  '  +  '  type= '+ str(type) )

            #     plt.savefig(os.path.join(cfg.plots_dir, f"{obj_id}.png"))
            #     plt.close()
               
                

        global_ii += 1


df = pd.DataFrame(data_2)
df.to_csv(cfg.long_csv, header=['SNID', 'Type', 'Field', 'Filter', 'Photo_z', 'Pk_Mag', 'Beta1', 'Beta2', 'Pk_Time', 'Err_z', 'Err_Mag',
                                      'Err_Beta1', 'Err_Beta2', 'Err_Time', 'Chi_2', 'DOF'], index=False)


print(f"Rows in {cfg.long_csv} (one per SN-band fit): {len(df)}")


Total unique SNIDs across all files: 81264
Progress: 0%  (0/81264)
Progress: 1%  (812/81264)
Progress: 2%  (1624/81264)
Progress: 3%  (2436/81264)
Progress: 4%  (3248/81264)
Progress: 5%  (4060/81264)
Progress: 6%  (4872/81264)
Progress: 7%  (5684/81264)
Progress: 8%  (6496/81264)
Progress: 9%  (7308/81264)
Progress: 10%  (8120/81264)
Progress: 11%  (8932/81264)
Progress: 12%  (9744/81264)
Progress: 13%  (10556/81264)
Progress: 14%  (11368/81264)
Progress: 15%  (12180/81264)
Progress: 16%  (12992/81264)
Progress: 17%  (13804/81264)
Progress: 18%  (14616/81264)
Progress: 19%  (15428/81264)
Progress: 20%  (16240/81264)
Progress: 21%  (17052/81264)
Progress: 22%  (17864/81264)
Progress: 23%  (18676/81264)
Progress: 24%  (19488/81264)
Progress: 25%  (20300/81264)
Progress: 26%  (21112/81264)
Progress: 27%  (21924/81264)
Progress: 28%  (22736/81264)
Progress: 29%  (23548/81264)
Progress: 30%  (24360/81264)
Progress: 31%  (25172/81264)
Progress: 32%  (25984/81264)
Progress: 33%  (26796/81264

### 6. Converting to wide-format

In [6]:
df_long = pd.read_csv(cfg.long_csv)

df_long['Filter'] = df_long['Filter'].astype(str)

# --- 2. Define Columns for Pivoting and Metadata ---

# 8 columns per filter to be spread out
value_cols = ['Pk_Mag', 'Beta1', 'Beta2', 'Chi_2', 'DOF', 'Pk_Time',
    'Err_Mag', 'Err_Beta1', 'Err_Beta2', 'Err_Time']

# Columns to keep as object metadata (should be consistent across all filters for an object)
meta_cols = ['Field', 'Type',  'Photo_z', 'Err_z']

# --- 3. Pivot the Data from Long to Wide Format ---

# Pivot on 'meta_id' to get one row per object, using 'filter' to spread the columns
pivoted_df = df_long.pivot(
    index='SNID',
    columns='Filter',
    values=value_cols
)

# --- 4. Flatten Multi-Index Columns and Rename ---

# Flatten the column names to the format 'filter_parameter' (e.g., 'g_alpha')
# The structure is (parameter_name, filter_name), so we reverse the order
pivoted_df.columns = [f'{col[1]}_{col[0]}' for col in pivoted_df.columns]
pivoted_df = pivoted_df.reset_index() # Move 'SNID' from index back to a column

# --- 5. Merge Metadata and Final Output ---

# Extract unique metadata rows and merge them back with the pivoted data
meta_unique_df = df_long[['SNID'] + meta_cols].drop_duplicates(subset=['SNID']).reset_index(drop=True)

# Merge the metadata onto the wide-format results
final_wide_df = meta_unique_df.merge(pivoted_df, on='SNID', how='right')

# 6. Save the final wide-format CSV file

final_wide_df.to_csv(cfg.wide_csv, index=False)


print(f"Final DataFrame has {len(final_wide_df)} rows (one per object).")



Final DataFrame has 17991 rows (one per object).


### 7. Converting apparent magnitudes to absolute magnitudes


In [7]:
cosmo = config.cosmology.build()

df = pd.read_csv(cfg.wide_csv)

d = cosmo.luminosity_distance(df['Photo_z'].to_numpy(dtype=float)).to(u.pc).value

mag_cols = [c for c in df.columns if c.endswith('_Pk_Mag')]
mag = df[mag_cols].to_numpy(dtype=float)
mag_cal = mag + 5 - 5*np.log10(d)[:, None]

df_out = df.copy()
df_out[mag_cols] = mag_cal
df_out = df_out.rename(columns={c: c.replace('_Pk_Mag', '_Abs_Mag') for c in mag_cols})

df_out.to_csv(config.globals.abs_mag_csv, index=False)

print(f"Saved Absolute Magnitudes for {len(df_out)} SNe to {config.globals.abs_mag_csv}")

Saved Absolute Magnitudes for 17991 SNe to Files_SD/Output_Final.csv
